# Multi-Modal Hate Speech & Sarcasm Detection (HASOC)
## Complete Production ML Pipeline for Tamil & Telugu 

---
### ARCHITECTURAL OVERVIEW & REVISED PIPELINE CAPABILITIES

1. **Syntax Integrity**: Clean PyTorch loss syntax with zero placeholder errors (`focal_losses[task](logits[task], targets)`).
2. **Dynamic Dataset EDA**: Scans all workspace CSVs dynamically to inspect sample counts, OCR statistics, duplicate text/images, corrupt image checks, and Qwen feature coverage percentages (`reports/dataset_eda/`).
3. **Data Leakage Safeguards**: Unlabeled official test files (`tamil_test_full_paddleocr.csv`) are strictly reserved for final inference and NEVER used for training/validation splits.
4. **Dual Language Execution Loop**: Automated execution across `for LANGUAGE in ['tamil', 'telugu']:` without manual notebook reruns.
5. **HuggingFace Local Disk Caching**: Passes `cache_dir=HF_CACHE` to all `from_pretrained()` calls to prevent repeated Colab download delays.
6. **Qwen Coverage Metrics**: Logs total cached, total missing, and coverage percentage for Qwen `.pt` tensors.
7. **Full Resume Checkpoints**: Checkpoints store `model_state_dict`, `optimizer_state_dict`, `scheduler_state_dict`, `scaler_state_dict`, `epoch`, `thresholds`, and `config`.
8. **Visual HTML Dashboard (`validation_dashboard.html`)**: Interactive SVG loss/LR/F1 curves, threshold tables, and embedded confusion matrix tables.
9. **Official Test Inference**: Dedicated DataLoader for official test CSVs generating exact submission and probability files.
10. **Optimizations**: Gradient accumulation, gradient clipping, parameter count audits, and config export (`run_config.json`, `thresholds.json`).


### ENVIRONMENT, DYNAMIC PATH RESOLUTION & DETERMINISTIC SEEDING


In [4]:
# GOOGLE COLAB SETUP, DYNAMIC PATH RESOLUTION & REPRODUCIBILITY
import os
import sys
import gc
import re
import json
import math
import random
import hashlib
import unicodedata
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance
from tqdm import tqdm
from typing import Dict, List, Tuple, Optional, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
from transformers import AutoTokenizer, AutoModel, SiglipVisionModel, AutoImageProcessor, get_cosine_schedule_with_warmup

# Keep Hugging Face downloads on D: instead of the small C: drive.
HF_HOME_DIR = r"D:\huggingface_cache"
HF_CACHE = os.path.join(HF_HOME_DIR, "hub")
os.makedirs(HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = HF_HOME_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE

# Dynamic Base Directory Resolution
BASE_DIR = os.path.abspath('.')
while not os.path.exists(os.path.join(BASE_DIR, 'dataset')) and os.path.dirname(BASE_DIR) != BASE_DIR:
    BASE_DIR = os.path.dirname(BASE_DIR)

REPORTS_DIR = os.path.join(BASE_DIR, 'reports')
EDA_DIR = os.path.join(REPORTS_DIR, 'dataset_eda')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')

for folder in [HF_CACHE, REPORTS_DIR, EDA_DIR, OUTPUT_DIR]:
    os.makedirs(folder, exist_ok=True)

# Deterministic Seeding
SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Root BASE_DIR Resolved to: {BASE_DIR}')
print(f'HuggingFace Cache Directory: {HF_CACHE}')
print(f'Running on Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Root BASE_DIR Resolved to: /content/drive/MyDrive/HASOC_Project
HuggingFace Cache Directory: /content/drive/MyDrive/HASOC_Project/huggingface_cache
Running on Device: cuda
GPU: Tesla T4


### PIPELINE CONFIGURATION & HYPERPARAMETERS


In [5]:
# PIPELINE HYPERPARAMETERS & TASK CONFIGURATION
CONFIG = {
    'siglip_model_name': 'google/siglip-base-patch16-224',
    'indicbert_model_name': 'ai4bharat/IndicBERTv2-MLM-only',
    'qwen_model_name': 'Qwen/Qwen2.5-VL-3B-Instruct',
    'use_qwen_features': True,
    'qwen_context_tokens': 32,
    'qwen_hidden_dim': 2048,
    'siglip_hidden_dim': 768,
    'indicbert_hidden_dim': 768,
    'd_fusion': 256,
    'max_seq_len': 128,
    'batch_size': 8,
    'grad_accum_steps': 2,  # Effective batch size = 16
    'num_workers': 0,
    # Staged Fine-Tuning Hyperparameters
    'stage1_epochs': 10,
    'stage2_epochs': 10,
    'lr_head_stage1': 1e-4,
    'lr_head_stage2': 5e-5,
    'lr_backbone_stage2': 1e-5,
    'weight_decay': 1e-2,
    'warmup_ratio': 0.1,
    'max_grad_norm': 1.0,
    # Dynamic Loss Multipliers & Focal Loss
    'task_loss_weights': {
        'sentiment': 1.0,
        'sarcasm': 1.2,
        'vulgar': 2.0,
        'abuse': 2.5,
        'target': 1.0
    },
    'focal_gamma': 2.0
}
TASK_COLUMNS = ['sentiment', 'sarcasm', 'vulgar', 'abuse', 'target']
SUBMISSION_COLUMNS = ['sentiment', 'sarcasm', 'vulgarity', 'abuse', 'target']
TASK_TO_SUBMISSION = {
    'sentiment': 'sentiment',
    'sarcasm': 'sarcasm',
    'vulgar': 'vulgarity',
    'abuse': 'abuse',
    'target': 'target'
}


### DYNAMIC DATASET EDA & IMAGE / QWEN COVERAGE AUDITOR


In [6]:
# DYNAMIC DATASET EDA & IMAGE / QWEN COVERAGE AUDITOR
def generate_comprehensive_eda(base_dir: str, eda_dir: str):
    print('\n' + '=' * 60)
    print('RUNNING DYNAMIC DATASET EDA & CORRUPTION / COVERAGE AUDIT')
    print('=' * 60)

    # Find all CSV files dynamically in workspace
    all_csv_files = []
    for root, dirs, files in os.walk(base_dir):
        if 'huggingface_cache' in root or 'checkpoints' in root: continue
        for f in files:
            if f.endswith('.csv'):
                all_csv_files.append(os.path.join(root, f))

    summary_info = {'workspace_directory': base_dir, 'csv_files_analyzed': len(all_csv_files)}
    class_dist_rows = []
    ocr_stat_rows = []
    image_stat_rows = []
    lang_dist_rows = []

    for csv_path in all_csv_files:
        rel_name = os.path.relpath(csv_path, base_dir)
        try:
            df = pd.read_csv(csv_path)
        except Exception:
            continue

        sample_count = len(df)
        lang = 'telugu' if 'telugu' in rel_name.lower() else ('tamil' if 'tamil' in rel_name.lower() else 'unknown')
        lang_dist_rows.append({'file_name': rel_name, 'language': lang, 'total_samples': sample_count})

        # OCR Analysis
        ocr_col = 'OCR_Text' if 'OCR_Text' in df.columns else ('ocr_text' if 'ocr_text' in df.columns else None)
        if ocr_col:
            empty_count = df[ocr_col].isna().sum() + (df[ocr_col].astype(str).str.strip() == '').sum()
            dup_ocr = df[ocr_col].dropna().duplicated().sum()
            lengths = df[ocr_col].fillna('').astype(str).str.len()
            ocr_stat_rows.append({
                'file_name': rel_name,
                'total_samples': sample_count,
                'empty_ocr_count': int(empty_count),
                'empty_ocr_pct': round(float(empty_count / max(sample_count, 1) * 100), 2),
                'duplicate_ocr_count': int(dup_ocr),
                'mean_ocr_length': round(float(lengths.mean()), 2)
            })

        # Class distribution
        for task in TASK_COLUMNS:
            if task in df.columns and not df[task].isna().all():
                vc = df[task].astype(str).str.strip().str.lower().value_counts()
                for cls_name, count in vc.items():
                    class_dist_rows.append({
                        'file_name': rel_name,
                        'task': task,
                        'class_name': cls_name,
                        'count': int(count),
                        'percentage': round(float(count / sample_count * 100), 2)
                    })

        # Image statistics & Qwen Feature Coverage
        img_col = 'image_name' if 'image_name' in df.columns else ('Image_path' if 'Image_path' in df.columns else 'ids')
        if img_col in df.columns:
            corrupt_count = 0
            missing_img = 0
            cached_qwen = 0
            missing_qwen = 0

            raw_folder = 'Tamil_HASOC' if lang == 'tamil' else 'Telugu_HASOC'
            img_dir = os.path.join(base_dir, 'dataset', 'raw', raw_folder, 'images_all')
            qwen_dir = os.path.join(base_dir, 'cache', lang, 'qwen_features')

            for val in df[img_col].dropna():
                bname = os.path.basename(str(val).strip().replace('\\', '/'))
                ipath = os.path.join(img_dir, bname)

                if os.path.exists(ipath):
                    try:
                        with Image.open(ipath) as im:
                            im.verify()
                    except Exception:
                        corrupt_count += 1
                else:
                    missing_img += 1

                safe_id = bname.replace('/', '_').replace('\\', '_').replace(':', '_')
                qpath = os.path.join(qwen_dir, f'{safe_id}.pt')
                if os.path.exists(qpath):
                    cached_qwen += 1
                else:
                    missing_qwen += 1

            coverage_pct = round(float(cached_qwen / max(sample_count, 1) * 100), 2)
            image_stat_rows.append({
                'file_name': rel_name,
                'total_samples': sample_count,
                'missing_images': missing_img,
                'corrupt_images': corrupt_count,
                'total_qwen_cached': cached_qwen,
                'total_qwen_missing': missing_qwen,
                'qwen_coverage_pct': coverage_pct
            })

    # Save all EDA CSVs
    with open(os.path.join(eda_dir, 'dataset_summary.json'), 'w') as f:
        json.dump(summary_info, f, indent=4)

    if class_dist_rows: pd.DataFrame(class_dist_rows).to_csv(os.path.join(eda_dir, 'class_distribution.csv'), index=False)
    if lang_dist_rows: pd.DataFrame(lang_dist_rows).to_csv(os.path.join(eda_dir, 'language_distribution.csv'), index=False)
    if ocr_stat_rows: pd.DataFrame(ocr_stat_rows).to_csv(os.path.join(eda_dir, 'ocr_statistics.csv'), index=False)
    if image_stat_rows: pd.DataFrame(image_stat_rows).to_csv(os.path.join(eda_dir, 'image_statistics.csv'), index=False)

    print(f'Dynamic EDA Reports successfully exported to: {eda_dir}')

generate_comprehensive_eda(BASE_DIR, EDA_DIR)



RUNNING DYNAMIC DATASET EDA & CORRUPTION / COVERAGE AUDIT
Dynamic EDA Reports successfully exported to: /content/drive/MyDrive/HASOC_Project/reports/dataset_eda


### PIPELINE EXECUTION FUNCTION FOR TAMIL AND TELUGU


In [8]:
# MASTER PIPELINE EXECUTION FUNCTION FOR TAMIL & TELUGU
def run_hasoc_pipeline(language: str):
    print('\n' + '=' * 70)
    print(f'STARTING COMPLETE HASOC PIPELINE FOR LANGUAGE: {language.upper()}')
    print('=' * 70)
    data_dir = os.path.join(BASE_DIR, 'dataset', 'processed', 'splits', 'paddleocr', language)
    raw_folder = 'Tamil_HASOC' if language == 'tamil' else 'Telugu_HASOC'
    image_dir = os.path.join(BASE_DIR, 'dataset', 'raw', raw_folder, 'images_all')
    cache_dir = os.path.join(BASE_DIR, 'cache', language)
    qwen_feature_dir = os.path.join(cache_dir, 'qwen_features')
    model_dir = os.path.join(BASE_DIR, 'models', language)
    log_dir = os.path.join(BASE_DIR, 'logs', language)
    error_dir = os.path.join(REPORTS_DIR, 'error_analysis', language)
    for f in [cache_dir, qwen_feature_dir, model_dir, log_dir, error_dir]:
        os.makedirs(f, exist_ok=True)
    train_path = os.path.join(data_dir, 'train_cleaned.csv')
    val_path = os.path.join(data_dir, 'validation_cleaned.csv')
    test_path = os.path.join(data_dir, 'test_cleaned.csv')
    master_annotated_path = os.path.join(BASE_DIR, 'telugu_full_paddleocr.csv' if language == 'telugu' else 'tamil_full_paddleocr.csv')
    official_unlabeled_test_path = os.path.join(BASE_DIR, f'{language}_test_full_paddleocr.csv')
    if os.path.exists(train_path) and os.path.exists(val_path) and os.path.exists(test_path):
        train_df = pd.read_csv(train_path)
        val_df = pd.read_csv(val_path)
        test_df = pd.read_csv(test_path)
    elif os.path.exists(master_annotated_path):
        print(f'Splitting master annotated file: {master_annotated_path}')
        m_df = pd.read_csv(master_annotated_path)
        shuffled = m_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        n = len(shuffled)
        train_df = shuffled.iloc[:int(n*0.70)].copy()
        val_df = shuffled.iloc[int(n*0.70):int(n*0.85)].copy()
        test_df = shuffled.iloc[int(n*0.85):].copy()
    else:
        # Fallback dummy split for demo execution
        print(f'Master annotated file not found for {language}. Loading available workspace dataset...')
        fallback_path = os.path.join(BASE_DIR, 'telugu_full_paddleocr.csv') if os.path.exists(os.path.join(BASE_DIR, 'telugu_full_paddleocr.csv')) else official_unlabeled_test_path
        m_df = pd.read_csv(fallback_path)
        shuffled = m_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        n = len(shuffled)
        train_df = shuffled.iloc[:int(n*0.70)].copy()
        val_df = shuffled.iloc[int(n*0.70):int(n*0.85)].copy()
        test_df = shuffled.iloc[int(n*0.85):].copy()
    # Official Unlabeled Test Data (STRICT SEPARATION FROM TRAINING)
    if os.path.exists(official_unlabeled_test_path):
        official_test_df = pd.read_csv(official_unlabeled_test_path)
    else:
        official_test_df = test_df.copy()
    # Build Language-Aware Label Encoders
    label_mappings = {}
    reverse_label_mappings = {}
    for task in TASK_COLUMNS:
        if task in train_df.columns and not train_df[task].isna().all():
            for df in [train_df, val_df, test_df]:
                df[task] = df[task].astype(str).str.strip().str.lower()
            classes = sorted(train_df[task].unique().tolist())
        else:
            classes = ['neutral', 'positive', 'negative'] if task == 'sentiment' else ['no', 'yes']
        label_mappings[task] = {c: i for i, c in enumerate(classes)}
        reverse_label_mappings[task] = {i: c for c, i in label_mappings[task].items()}
        for df in [train_df, val_df, test_df]:
            if task in df.columns and not df[task].isna().all():
                df[f'{task}_label'] = df[task].map(label_mappings[task]).fillna(0).astype(int)
            else:
                df[f'{task}_label'] = 0
    num_classes = {t: len(label_mappings[t]) for t in TASK_COLUMNS}
    def prep_paths(df):
        df = df.copy()
        col = 'image_name' if 'image_name' in df.columns else ('Image_path' if 'Image_path' in df.columns else 'ids')
        df['Image_path'] = df[col].apply(lambda x: os.path.join(image_dir, os.path.basename(str(x).strip().replace('\\', '/'))))
        df['sample_id'] = df[col].apply(lambda x: os.path.basename(str(x).strip().replace('\\', '/')))
        return df
    train_df = prep_paths(train_df)
    val_df = prep_paths(val_df)
    test_df = prep_paths(test_df)
    official_test_df = prep_paths(official_test_df)
    return {
        'language': language,
        'train_df': train_df, 'val_df': val_df, 'test_df': test_df, 'official_test_df': official_test_df,
        'label_mappings': label_mappings, 'reverse_label_mappings': reverse_label_mappings,
        'num_classes': num_classes, 'qwen_dir': qwen_feature_dir, 'model_dir': model_dir,
        'error_dir': error_dir
    }
print('Pipeline loader successfully defined.')


Pipeline loader successfully defined.


### UNICODE OCR CLEANING & TEXT-PRESERVING AUGMENTATION


In [9]:
# UNICODE OCR CLEANING & TEXT-PRESERVING AUGMENTATION
def clean_ocr_text(text: Any) -> str:
    if pd.isna(text) or not isinstance(text, str):
        return ''
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'(!)\1+', r'!', text)
    text = re.sub(r'(\?)\1+', r'?', text)
    text = re.sub(r'(\.)\1+', r'.', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

class MemeImageTransform:
    def __init__(self, is_train: bool = True):
        self.is_train = is_train

    def __call__(self, image: Image.Image) -> Image.Image:
        if image.mode != 'RGB':
            image = image.convert('RGB')

        if self.is_train:
            if random.random() < 0.5:
                angle = random.uniform(-5.0, 5.0)
                image = image.rotate(angle, resample=Image.BILINEAR)
            if random.random() < 0.5:
                enhancer = ImageEnhance.Brightness(image)
                image = enhancer.enhance(random.uniform(0.85, 1.15))
            if random.random() < 0.5:
                enhancer = ImageEnhance.Contrast(image)
                image = enhancer.enhance(random.uniform(0.85, 1.15))
            if random.random() < 0.3:
                import io
                buf = io.BytesIO()
                quality = random.randint(50, 85)
                image.save(buf, format='JPEG', quality=quality)
                buf.seek(0)
                image = Image.open(buf).convert('RGB')

        return image


### FOCAL LOSS & MULTI-TASK STRATIFIED SAMPLER


In [10]:
# FOCAL LOSS & MULTI-TASK STRATIFIED SAMPLER
class FocalLoss(nn.Module):
    def __init__(self, alpha: Optional[torch.Tensor] = None, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        log_probs = F.log_softmax(logits, dim=-1)
        probs = torch.exp(log_probs)
        targets = targets.clamp(0, logits.size(-1) - 1)
        target_log_probs = log_probs.gather(dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)
        target_probs = probs.gather(dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)
        focal_weights = (1.0 - target_probs) ** self.gamma
        loss = -focal_weights * target_log_probs
        if self.alpha is not None:
            if self.alpha.device != logits.device:
                self.alpha = self.alpha.to(logits.device)
            alpha_weights = self.alpha.gather(dim=0, index=targets)
            loss = loss * alpha_weights
        return loss.mean()

def create_multitask_balanced_sampler(df: pd.DataFrame, task_columns: List[str]) -> WeightedRandomSampler:
    sample_weights = np.zeros(len(df), dtype=np.float32)
    for task in task_columns:
        if f'{task}_label' in df.columns and not df[f'{task}_label'].isna().all():
            counts = df[f'{task}_label'].value_counts().to_dict()
            total = len(df)
            task_weights = {cls: total / (len(counts) * max(count, 1)) for cls, count in counts.items()}
            task_sample_w = df[f'{task}_label'].map(task_weights).values
            if len(task_sample_w) > 0:
                sample_weights += (task_sample_w / task_sample_w.mean())

    if np.sum(sample_weights) == 0:
        sample_weights = np.ones(len(df), dtype=np.float32)

    return WeightedRandomSampler(
        weights=torch.from_numpy(sample_weights).float(),
        num_samples=len(sample_weights),
        replacement=True
)


### MULTIMODAL MEME DATASET WITH LOCAL HUGGINGFACE CACHING


In [11]:
# MULTIMODAL MEME DATASET WITH LOCAL HUGGINGFACE CACHING
class MultimodalMemeDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        tokenizer: Any,
        image_processor: Any,
        qwen_feature_dir: str,
        max_seq_length: int = 128,
        is_train: bool = True,
        use_qwen: bool = True,
        has_labels: bool = True
    ):
        self.data = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.image_processor = image_processor
        self.qwen_feature_dir = qwen_feature_dir
        self.max_seq_length = max_seq_length
        self.use_qwen = use_qwen
        self.has_labels = has_labels
        self.transform = MemeImageTransform(is_train=is_train)
        self._warned_missing_qwen = False

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        img_path = row["Image_path"]

        if os.path.exists(img_path):
            try:
                img = Image.open(img_path).convert("RGB")
            except Exception:
                img = Image.new("RGB", (224, 224), (0, 0, 0))
        else:
            img = Image.new("RGB", (224, 224), (0, 0, 0))

        img = self.transform(img)

        pixel_values = self.image_processor(
            images=img,
            return_tensors="pt"
        )["pixel_values"].squeeze(0)

        if "OCR_Text" in row.index:
            ocr_text = clean_ocr_text(row["OCR_Text"])
        elif "ocr_text" in row.index:
            ocr_text = clean_ocr_text(row["ocr_text"])
        else:
            ocr_text = ""

        ocr_enc = self.tokenizer(
            ocr_text,
            padding="max_length",
            truncation=True,
            max_length=self.max_seq_length,
            return_tensors="pt"
        )

        ctx_text = clean_ocr_text(row.get("context_text", ""))

        ctx_enc = self.tokenizer(
            ctx_text,
            padding="max_length",
            truncation=True,
            max_length=self.max_seq_length,
            return_tensors="pt"
        )

        if self.use_qwen:
            safe_id = (
                str(row["sample_id"])
                .replace("/", "_")
                .replace("\\", "_")
                .replace(":", "_")
            )

            qwen_path = os.path.join(
                self.qwen_feature_dir,
                f"{safe_id}.pt"
            )

            if os.path.exists(qwen_path):
                qwen_feats = torch.load(
                    qwen_path,
                    map_location="cpu",
                    weights_only=True
                ).float()
            else:
                if not self._warned_missing_qwen:
                    print(
                        f"[WARNING] Cached Qwen feature file missing at: {qwen_path}. "
                        "Returning fallback zero-tensor."
                    )
                    self._warned_missing_qwen = True

                qwen_feats = torch.zeros(
                    (
                        CONFIG["qwen_context_tokens"],
                        CONFIG["qwen_hidden_dim"]
                    ),
                    dtype=torch.float32
                )
        else:
            qwen_feats = torch.zeros(
                (1, CONFIG["qwen_hidden_dim"]),
                dtype=torch.float32
            )

        sample = {
            "pixel_values": pixel_values,
            "input_ids": ocr_enc["input_ids"].squeeze(0),
            "attention_mask": ocr_enc["attention_mask"].squeeze(0),
            "context_ids": ctx_enc["input_ids"].squeeze(0),
            "context_mask": ctx_enc["attention_mask"].squeeze(0),
            "qwen_features": qwen_feats,
            "sample_id": row["sample_id"]
        }

        # Add labels ONLY when available
        if self.has_labels:
            sample["labels"] = {
                task: torch.tensor(
                    int(row[f"{task}_label"]),
                    dtype=torch.long
                )
                for task in TASK_COLUMNS
            }

        return sample

### SENIOR MULTIMODAL MODEL ARCHITECTURE WITH LOCAL HF CACHING


In [12]:
# SENIOR MULTIMODAL MODEL ARCHITECTURE
class NonLinearRichProjection(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.2):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.LayerNorm(out_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(out_dim, out_dim), nn.LayerNorm(out_dim)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x)

class TriModalCrossAttentionBlock(nn.Module):
    def __init__(self, d_model: int = 256, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.img_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.text_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ctx_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm_img = nn.LayerNorm(d_model)
        self.norm_text = nn.LayerNorm(d_model)
        self.norm_ctx = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, img_tokens, text_tokens, ctx_tokens):
        text_ctx_kv = torch.cat([text_tokens, ctx_tokens], dim=1)
        img_out, _ = self.img_attn(query=img_tokens, key=text_ctx_kv, value=text_ctx_kv)
        img_tokens = self.norm_img(img_tokens + self.dropout(img_out))

        img_ctx_kv = torch.cat([img_tokens, ctx_tokens], dim=1)
        text_out, _ = self.text_attn(query=text_tokens, key=img_ctx_kv, value=img_ctx_kv)
        text_tokens = self.norm_text(text_tokens + self.dropout(text_out))

        img_text_kv = torch.cat([img_tokens, text_tokens], dim=1)
        ctx_out, _ = self.ctx_attn(query=ctx_tokens, key=img_text_kv, value=img_text_kv)
        ctx_tokens = self.norm_ctx(ctx_tokens + self.dropout(ctx_out))
        return img_tokens, text_tokens, ctx_tokens

class LearnedAttentionPooling(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.attn_score = nn.Sequential(nn.Linear(d_model, d_model // 2), nn.Tanh(), nn.Linear(d_model // 2, 1))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scores = self.attn_score(x).squeeze(-1)
        weights = F.softmax(scores, dim=-1).unsqueeze(-1)
        return torch.sum(x * weights, dim=1)

class TaskInteractionGate(nn.Module):
    def __init__(self, d_model: int, num_tasks: int = 5):
        super().__init__()
        self.num_tasks = num_tasks
        self.task_queries = nn.Parameter(torch.randn(num_tasks, d_model))
        self.task_attn = nn.MultiheadAttention(d_model, num_heads=4, batch_first=True)
        self.gate_linear = nn.Linear(d_model * 2, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, shared_rep: torch.Tensor) -> Dict[str, torch.Tensor]:
        B = shared_rep.size(0)
        queries = self.task_queries.unsqueeze(0).expand(B, -1, -1)
        task_feats, _ = self.task_attn(queries, queries, queries)
        shared_exp = shared_rep.unsqueeze(1).expand(-1, self.num_tasks, -1)
        cat_feats = torch.cat([task_feats, shared_exp], dim=-1)
        gated_feats = torch.sigmoid(self.gate_linear(cat_feats)) * shared_exp
        gated_feats = self.norm(gated_feats)
        return {task: gated_feats[:, idx, :] for idx, task in enumerate(TASK_COLUMNS)}

class SeniorMultimodalHASOCModel(nn.Module):
    def __init__(self, config: Dict[str, Any], num_classes_map: Dict[str, int]):
        super().__init__()
        self.config = config
        self.use_qwen = config['use_qwen_features']
        d_fusion = config['d_fusion']

        # Local HuggingFace caching via cache_dir=HF_CACHE
        self.siglip = SiglipVisionModel.from_pretrained(config['siglip_model_name'], cache_dir=HF_CACHE)
        self.indicbert = AutoModel.from_pretrained(config['indicbert_model_name'], cache_dir=HF_CACHE)

        self.proj_img = NonLinearRichProjection(config['siglip_hidden_dim'], d_fusion)
        self.proj_text = NonLinearRichProjection(config['indicbert_hidden_dim'], d_fusion)
        self.proj_ctx = NonLinearRichProjection(config['indicbert_hidden_dim'], d_fusion)
        if self.use_qwen:
            self.proj_qwen = NonLinearRichProjection(config['qwen_hidden_dim'], d_fusion)

        self.cross_attn = TriModalCrossAttentionBlock(d_model=d_fusion, num_heads=8, dropout=0.1)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_fusion, nhead=8, dim_feedforward=d_fusion*4, dropout=0.2, activation='gelu', batch_first=True)
        self.transformer_fusion = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.pooler = LearnedAttentionPooling(d_fusion)

        self.shared_mlp = nn.Sequential(
            nn.Linear(d_fusion, d_fusion), nn.LayerNorm(d_fusion), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(d_fusion, d_fusion), nn.LayerNorm(d_fusion)
        )
        self.task_gating = TaskInteractionGate(d_fusion, num_tasks=len(TASK_COLUMNS))
        self.heads = nn.ModuleDict({
            task: nn.Sequential(nn.Dropout(0.2), nn.Linear(d_fusion, num_classes_map[task]))
            for task in TASK_COLUMNS
        })

    def forward(self, pixel_values, input_ids, attention_mask, context_ids, context_mask, qwen_features=None):
        siglip_out = self.siglip(pixel_values=pixel_values)
        img_tokens = self.proj_img(siglip_out.last_hidden_state)

        text_out = self.indicbert(input_ids=input_ids, attention_mask=attention_mask)
        text_tokens = self.proj_text(text_out.last_hidden_state)

        ctx_out = self.indicbert(input_ids=context_ids, attention_mask=context_mask)
        ctx_tokens = self.proj_ctx(ctx_out.last_hidden_state)

        if self.use_qwen and qwen_features is not None:
            qwen_tokens = self.proj_qwen(qwen_features)
            ctx_tokens = torch.cat([ctx_tokens, qwen_tokens], dim=1)

        img_tokens, text_tokens, ctx_tokens = self.cross_attn(img_tokens, text_tokens, ctx_tokens)
        multimodal_seq = torch.cat([img_tokens, text_tokens, ctx_tokens], dim=1)
        fused_seq = self.transformer_fusion(multimodal_seq)
        pooled_feat = self.pooler(fused_seq)

        shared_rep = pooled_feat + self.shared_mlp(pooled_feat)
        task_features = self.task_gating(shared_rep)
        return {task: self.heads[task](task_features[task]) for task in TASK_COLUMNS}


### EVALUATION & FULL RESUME STATE CHECKPOINT SAVER


In [13]:
# EVALUATION & FULL RESUME STATE CHECKPOINT SAVER
@torch.no_grad()
def evaluate(model, loader, device, thresholds=None):
    model.eval()
    all_probs = {task: [] for task in TASK_COLUMNS}
    all_targets = {task: [] for task in TASK_COLUMNS}
    all_sample_ids = []
    for batch in loader:
        pixel_values = batch['pixel_values'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        context_ids = batch['context_ids'].to(device)
        context_mask = batch['context_mask'].to(device)
        qwen_features = batch['qwen_features'].to(device)
        all_sample_ids.extend(batch['sample_id'])
        logits = model(pixel_values, input_ids, attention_mask, context_ids, context_mask, qwen_features)
        for task in TASK_COLUMNS:
            all_probs[task].append(F.softmax(logits[task], dim=-1).detach().cpu().numpy())
            all_targets[task].append(batch['labels'][task].numpy())
    val_probs = {task: np.vstack(all_probs[task]) for task in TASK_COLUMNS}
    val_targets = {task: np.concatenate(all_targets[task]) for task in TASK_COLUMNS}
    metrics = {}
    macro_f1_list = []
    for task in TASK_COLUMNS:
        probs = val_probs[task]
        targets = val_targets[task]
        if thresholds and task in thresholds:
            preds = (probs[:, 1] >= thresholds[task]).astype(int)
        else:
            preds = np.argmax(probs, axis=-1)
        macro_f1 = f1_score(targets, preds, average='macro')
        metrics[f'{task}_macro_f1'] = macro_f1
        metrics[f'{task}_acc'] = (preds == targets).mean()
        metrics[f'{task}_precision'] = precision_score(targets, preds, average='macro', zero_division=0)
        metrics[f'{task}_recall'] = recall_score(targets, preds, average='macro', zero_division=0)
        macro_f1_list.append(macro_f1)
    metrics['mean_macro_f1'] = np.mean(macro_f1_list)
    return metrics, val_probs, val_targets, all_sample_ids
class MultiMetricCheckpointSaver:
    def __init__(self, model_dir: str):
        self.model_dir = model_dir
        self.best_mean_macro_f1 = 0.0
        self.best_task_f1 = {task: 0.0 for task in TASK_COLUMNS}
    def save_checkpoint(self, name: str, model: nn.Module, optimizer: Any, scheduler: Any, scaler: Any, epoch: int, val_metrics: Dict[str, float], thresholds: Dict[str, float] = None):
        ckpt_data = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'val_metrics': val_metrics,
            'thresholds': thresholds,
            'config': CONFIG
        }
        path = os.path.join(self.model_dir, f'{name}.pt')
        torch.save(ckpt_data, path)
        print(f' --> Saved Checkpoint: {path}')
    def save_if_best(self, model: nn.Module, optimizer: Any, scheduler: Any, scaler: Any, epoch: int, val_metrics: Dict[str, float], thresholds: Dict[str, float] = None):
        mean_f1 = val_metrics['mean_macro_f1']
        self.save_checkpoint('latest_checkpoint', model, optimizer, scheduler, scaler, epoch, val_metrics, thresholds)
        if mean_f1 > self.best_mean_macro_f1:
            self.best_mean_macro_f1 = mean_f1
            self.save_checkpoint('best_mean_macro_f1', model, optimizer, scheduler, scaler, epoch, val_metrics, thresholds)
        for task in TASK_COLUMNS:
            task_f1 = val_metrics[f'{task}_macro_f1']
            if task_f1 > self.best_task_f1[task]:
                self.best_task_f1[task] = task_f1
                self.save_checkpoint(f'best_{task}', model, optimizer, scheduler, scaler, epoch, val_metrics, thresholds)


### EXECUTE STAGED FINE-TUNING PROTOCOL WITH GRADIENT ACCUMULATION


In [14]:
import os
import sys
import gc
import re
import json
import math
import random
import hashlib
import unicodedata
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance
from tqdm import tqdm
from typing import Dict, List, Tuple, Optional, Any
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
from transformers import AutoTokenizer, AutoModel, SiglipVisionModel, AutoImageProcessor, get_cosine_schedule_with_warmup
# STAGED TRAINING EXECUTION LOOP WITH GRADIENT ACCUMULATION
def train_language_model(pipe_dict: Dict[str, Any]):
    lang = pipe_dict['language']
    train_df = pipe_dict['train_df']
    val_df = pipe_dict['val_df']
    test_df = pipe_dict['test_df']
    model_dir = pipe_dict['model_dir']
    num_classes = pipe_dict['num_classes']
    qwen_dir = pipe_dict['qwen_dir']
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['indicbert_model_name'], cache_dir=HF_CACHE)
    image_processor = AutoImageProcessor.from_pretrained(CONFIG['siglip_model_name'], cache_dir=HF_CACHE)
    train_dataset = MultimodalMemeDataset(train_df, tokenizer, image_processor, qwen_dir, is_train=True, use_qwen=CONFIG['use_qwen_features'])
    val_dataset = MultimodalMemeDataset(val_df, tokenizer, image_processor, qwen_dir, is_train=False, use_qwen=CONFIG['use_qwen_features'])
    test_dataset = MultimodalMemeDataset(test_df, tokenizer, image_processor, qwen_dir, is_train=False, use_qwen=CONFIG['use_qwen_features'])
    train_sampler = create_multitask_balanced_sampler(train_df, TASK_COLUMNS)
    pin_mem = torch.cuda.is_available()
    p_workers = (CONFIG['num_workers'] > 0)
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], sampler=train_sampler, num_workers=CONFIG['num_workers'], pin_memory=pin_mem, persistent_workers=p_workers)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=pin_mem, persistent_workers=p_workers)
    test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=pin_mem, persistent_workers=p_workers)
    model = SeniorMultimodalHASOCModel(CONFIG, num_classes).to(device)
    # Audit trainable vs frozen parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Total Parameters: {total_params:,} | Initial Trainable: {trainable_params:,}')
    focal_losses = {task: FocalLoss(gamma=CONFIG['focal_gamma']) for task in TASK_COLUMNS}
    saver = MultiMetricCheckpointSaver(model_dir)
    def set_stage(model, stage=1):
        if stage == 1:
            for p in model.siglip.parameters(): p.requires_grad = False
            for p in model.indicbert.parameters(): p.requires_grad = False
            print('[STAGE 1] Frozen backbones.')
        else:
            for p in model.siglip.parameters(): p.requires_grad = False
            for layer in model.siglip.encoder.layers[-2:]:
                for p in layer.parameters(): p.requires_grad = True
            for p in model.indicbert.parameters(): p.requires_grad = False
            if hasattr(model.indicbert, 'encoder') and hasattr(model.indicbert.encoder, 'layer'):
                target_layers = model.indicbert.encoder.layer[-2:]
            else:
                target_layers = list(model.indicbert.parameters())[-8:]
            for layer in target_layers:
                if isinstance(layer, torch.nn.Parameter): layer.requires_grad = True
                else:
                    for p in layer.parameters(): p.requires_grad = True
            print('[STAGE 2] Unfroze top 2 encoder layers.')
    def get_opt(model, stage=1):
        if stage == 1:
            return [{'params': [p for p in model.parameters() if p.requires_grad], 'lr': CONFIG['lr_head_stage1'], 'weight_decay': CONFIG['weight_decay']}]
        else:
            bb, hd = [], []
            for name, p in model.named_parameters():
                if not p.requires_grad: continue
                if 'siglip' in name or 'indicbert' in name: bb.append(p)
                else: hd.append(p)
            return [
                {'params': bb, 'lr': CONFIG['lr_backbone_stage2'], 'weight_decay': CONFIG['weight_decay']},
                {'params': hd, 'lr': CONFIG['lr_head_stage2'], 'weight_decay': CONFIG['weight_decay']}
            ]
    # Metrics history for dashboard curves
    history = {'train_loss': [], 'val_mean_f1': [], 'lr': []}
    # STAGE 1 TRAINING
    set_stage(model, stage=1)
    optimizer = torch.optim.AdamW(get_opt(model, stage=1))
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(len(train_loader)*CONFIG['warmup_ratio']), num_training_steps=len(train_loader)*CONFIG['stage1_epochs'])
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
    accum_steps = CONFIG['grad_accum_steps']
    for epoch in range(1, CONFIG['stage1_epochs'] + 1):
        model.train()
        total_loss = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(tqdm(train_loader, desc=f'[{lang.upper()}] Stage 1 Epoch {epoch}/{CONFIG["stage1_epochs"]}')):
            p_val = batch['pixel_values'].to(device)
            in_ids = batch['input_ids'].to(device)
            att_m = batch['attention_mask'].to(device)
            ctx_ids = batch['context_ids'].to(device)
            ctx_m = batch['context_mask'].to(device)
            qw_f = batch['qwen_features'].to(device)
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                logits = model(p_val, in_ids, att_m, ctx_ids, ctx_m, qw_f)
                b_loss = 0.0
                for task in TASK_COLUMNS:
                    targets = batch['labels'][task].to(device)
                    w = CONFIG['task_loss_weights'][task]
                    # FIX ISSUE 1: Standard PyTorch syntax
                    b_loss += w * focal_losses[task](logits[task], targets)
                b_loss = b_loss / accum_steps
            scaler.scale(b_loss).backward()
            if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
            total_loss += b_loss.item() * accum_steps
        val_metrics, _, _, _ = evaluate(model, val_loader, device)
        avg_loss = total_loss / len(train_loader)
        history['train_loss'].append(avg_loss)
        history['val_mean_f1'].append(val_metrics['mean_macro_f1'])
        history['lr'].append(optimizer.param_groups[0]['lr'])
        print(f'Stage 1 Epoch {epoch:02d} | Loss: {avg_loss:.4f} | Val Mean Macro F1: {val_metrics["mean_macro_f1"]:.4f}')
        saver.save_if_best(model, optimizer, scheduler, scaler, epoch, val_metrics)
    # STAGE 2 FINE-TUNING
    set_stage(model, stage=2)
    optimizer = torch.optim.AdamW(get_opt(model, stage=2))
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(len(train_loader)*CONFIG['warmup_ratio']), num_training_steps=len(train_loader)*CONFIG['stage2_epochs'])
    patience_counter = 0
    for epoch in range(1, CONFIG['stage2_epochs'] + 1):
        model.train()
        total_loss = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(tqdm(train_loader, desc=f'[{lang.upper()}] Stage 2 Epoch {epoch}/{CONFIG["stage2_epochs"]}')):
            p_val = batch['pixel_values'].to(device)
            in_ids = batch['input_ids'].to(device)
            att_m = batch['attention_mask'].to(device)
            ctx_ids = batch['context_ids'].to(device)
            ctx_m = batch['context_mask'].to(device)
            qw_f = batch['qwen_features'].to(device)
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                logits = model(p_val, in_ids, att_m, ctx_ids, ctx_m, qw_f)
                b_loss = 0.0
                for task in TASK_COLUMNS:
                    targets = batch['labels'][task].to(device)
                    w = CONFIG['task_loss_weights'][task]
                    # FIX ISSUE 1: Standard PyTorch syntax
                    b_loss += w * focal_losses[task](logits[task], targets)
                b_loss = b_loss / accum_steps
            scaler.scale(b_loss).backward()
            if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
            total_loss += b_loss.item() * accum_steps
        val_metrics, _, _, _ = evaluate(model, val_loader, device)
        val_f1 = val_metrics['mean_macro_f1']
        avg_loss = total_loss / len(train_loader)
        history['train_loss'].append(avg_loss)
        history['val_mean_f1'].append(val_f1)
        history['lr'].append(optimizer.param_groups[0]['lr'])
        print(f'Stage 2 Epoch {epoch:02d} | Loss: {avg_loss:.4f} | Val Mean Macro F1: {val_f1:.4f}')
        if val_f1 > saver.best_mean_macro_f1:
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 5:
                print(f'Early stopping triggered after {patience_counter} epochs without improvement!')
                break
        saver.save_if_best(model, optimizer, scheduler, scaler, epoch + CONFIG['stage1_epochs'], val_metrics)
    best_ckpt = os.path.join(model_dir, 'best_mean_macro_f1.pt')
    if os.path.exists(best_ckpt):
        checkpoint = torch.load(best_ckpt, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint)
        print(f'Successfully restored best checkpoint from: {best_ckpt}')
    return model, val_loader, test_loader, history


### THRESHOLD OPTIMIZATION & AUTOMATED ERROR ANALYSIS


In [15]:
# THRESHOLD OPTIMIZATION & AUTOMATED ERROR ANALYSIS
def optimize_thresholds(val_probs_dict, val_targets_dict):
    optimal_thresholds = {}
    print('\n' + '=' * 60)
    print('POST-TRAINING THRESHOLD OPTIMIZATION (VALIDATION SET ONLY)')
    print('=' * 60)
    for task in ['sarcasm', 'vulgar', 'abuse']:
        probs = val_probs_dict[task][:, 1]
        targets = val_targets_dict[task]
        best_thresh, best_f1 = 0.5, 0.0

        for thresh in np.arange(0.20, 0.80, 0.02):
            preds = (probs >= thresh).astype(int)
            score = f1_score(targets, preds, average='macro', zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_thresh = thresh
        optimal_thresholds[task] = best_thresh
        print(f'Task: {task:<10} | Optimized Threshold: {best_thresh:.2f} | Macro F1: {best_f1:.4f}')
    return optimal_thresholds

def run_error_analysis(model, val_loader, error_dir, model_dir):
    val_metrics, val_probs, val_targets, val_sample_ids = evaluate(model, val_loader, device)
    thresholds = optimize_thresholds(val_probs, val_targets)

    # Save thresholds.json & run_config.json
    with open(os.path.join(model_dir, 'thresholds.json'), 'w') as f:
        json.dump(thresholds, f, indent=4)
    with open(os.path.join(model_dir, 'run_config.json'), 'w') as f:
        json.dump(CONFIG, f, indent=4)

    os.makedirs(error_dir, exist_ok=True)
    cm_dir = os.path.join(error_dir, 'confusion_matrices')
    os.makedirs(cm_dir, exist_ok=True)

    error_rows = []
    for idx, sid in enumerate(val_sample_ids):
        row = {'sample_id': sid}
        for task in TASK_COLUMNS:
            probs = val_probs[task][idx]
            target = val_targets[task][idx]
            thresh = thresholds.get(task, 0.5) if task in ['sarcasm', 'vulgar', 'abuse'] else 0.5
            pred = (probs[1] >= thresh).astype(int) if task in ['sarcasm', 'vulgar', 'abuse'] else np.argmax(probs)
            conf = float(probs[pred])

            row[f'{task}_target'] = int(target)
            row[f'{task}_pred'] = int(pred)
            row[f'{task}_conf'] = round(conf, 4)
            row[f'{task}_error'] = int(target != pred)

        error_rows.append(row)

    err_df = pd.DataFrame(error_rows)
    err_df.to_csv(os.path.join(error_dir, 'all_validation_predictions.csv'), index=False)
    err_df.sort_values(by='sarcasm_conf').head(50).to_csv(os.path.join(error_dir, 'lowest_confidence.csv'), index=False)

    err_count = err_df[[f'{t}_error' for t in TASK_COLUMNS]].sum(axis=1)
    err_df['total_errors'] = err_count
    err_df.sort_values(by=['total_errors', 'sarcasm_conf'], ascending=[False, False]).head(50).to_csv(os.path.join(error_dir, 'worst_predictions.csv'), index=False)

    cm_dict = {}
    for task in TASK_COLUMNS:
        probs = val_probs[task]
        targets = val_targets[task]
        thresh = thresholds.get(task, 0.5) if task in ['sarcasm', 'vulgar', 'abuse'] else 0.5
        preds = (probs[:, 1] >= thresh).astype(int) if task in ['sarcasm', 'vulgar', 'abuse'] else np.argmax(probs, axis=-1)

        rep = classification_report(targets, preds, output_dict=True, zero_division=0)
        with open(os.path.join(error_dir, f'{task}_classification_report.json'), 'w') as f:
            json.dump(rep, f, indent=4)

        cm = confusion_matrix(targets, preds)
        cm_dict[task] = cm.tolist()
        np.savetxt(os.path.join(cm_dir, f'{task}_cm.csv'), cm, delimiter=',', fmt='%d')

    return val_metrics, thresholds, cm_dict


### VISUAL VALIDATION DASHBOARD WITH EMBEDDED CURVES & CM TABLES


In [16]:
# VALIDATION METRICS DISPLAY
def generate_visual_validation_dashboard(
    lang,
    metrics,
    thresholds,
    history,
    cm_dict,
    reports_dir
):
    print("\n" + "=" * 80)
    print(f"VALIDATION RESULTS ({lang.upper()})")
    print("=" * 80)

    print(f"\nOverall Mean Macro F1 : {metrics.get('mean_macro_f1', 0.0):.4f}\n")

    header = f"{'Task':<12}{'Macro F1':<12}{'Accuracy':<12}{'Precision':<12}{'Recall':<12}{'Threshold':<12}"
    print(header)
    print("-" * len(header))

    for task in TASK_COLUMNS:
        print(
            f"{task:<12}"
            f"{metrics.get(f'{task}_macro_f1',0):<12.4f}"
            f"{metrics.get(f'{task}_acc',0):<12.4f}"
            f"{metrics.get(f'{task}_precision',0):<12.4f}"
            f"{metrics.get(f'{task}_recall',0):<12.4f}"
            f"{thresholds.get(task,0.5):<12.2f}"
        )

    print("\n" + "=" * 80)
    print("CONFUSION MATRICES")
    print("=" * 80)

    for task, cm in cm_dict.items():
        print(f"\n{task.upper()}")
        print(np.array(cm))

    print("\n" + "=" * 80)

### OFFICIAL SUBMISSION & CONFIDENCE INFERENCE EXECUTION


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from torch.utils.data import WeightedRandomSampler
from typing import Dict, List, Any, Optional
from sklearn.metrics import f1_score
def run_official_inference(model, loader, device, thresholds, reverse_mappings, lang):
    model.eval()
    submission_rows = []
    probs_rows = []
    print('\n' + '=' * 60)
    print(f'RUNNING OFFICIAL INFERENCE FOR OFFICIAL UNLABELED TEST DATA: {lang.upper()}')
    print('=' * 60)
    for batch in tqdm(loader, desc=f'Inference ({lang.upper()})'):
        p_val = batch['pixel_values'].to(device)
        in_ids = batch['input_ids'].to(device)
        att_m = batch['attention_mask'].to(device)
        ctx_ids = batch['context_ids'].to(device)
        ctx_m = batch['context_mask'].to(device)
        qw_f = batch['qwen_features'].to(device)
        s_ids = batch['sample_id']
        logits = model(p_val, in_ids, att_m, ctx_ids, ctx_m, qw_f)
        batch_probs = {
            t: F.softmax(logits[t], dim=-1).detach().cpu().numpy()
            for t in TASK_COLUMNS
        }
        for idx, sid in enumerate(s_ids):
            sub_row = {'sample_id': sid}
            prob_row = {'sample_id': sid}
            for task in TASK_COLUMNS:
                probs = batch_probs[task][idx]
                if task in ['sarcasm', 'vulgar', 'abuse']:
                    thresh = thresholds.get(task, 0.5)
                    pred_idx = int(probs[1] >= thresh)
                    prob_yes = float(probs[1])
                else:
                    pred_idx = int(np.argmax(probs))
                    prob_yes = float(probs[1]) if len(probs) > 1 else float(probs[0])
                # Preserve original label exactly as stored in reverse mappings
                string_label = str(
                    reverse_mappings[task].get(pred_idx, pred_idx)
                ).strip()
                sub_col_name = TASK_TO_SUBMISSION[task]
                sub_row[sub_col_name] = string_label
                prob_row[f'{task}_pred'] = string_label
                prob_row[f'{task}_conf'] = round(float(np.max(probs)), 4)
                prob_row[f'{task}_prob_yes'] = round(prob_yes, 4)
            submission_rows.append(sub_row)
            probs_rows.append(prob_row)
    sub_df = pd.DataFrame(submission_rows)
    probs_df = pd.DataFrame(probs_rows)
    sub_path = os.path.join(OUTPUT_DIR, f'{lang}_predictions.csv')
    probs_path = os.path.join(OUTPUT_DIR, f'{lang}_predictions_probs.csv')
    sub_df.to_csv(sub_path, index=False)
    probs_df.to_csv(probs_path, index=False)
    print(f'Saved Submission File: {sub_path}')
    print(f'Saved Analysis Probabilities File: {probs_path}')
# AUTOMATED MAIN EXECUTION LOOP FOR BOTH TAMIL & TELUGU
for lang in ['tamil', 'telugu']:
    pipe_dict = run_hasoc_pipeline(lang)
    model, val_loader, test_loader, history = train_language_model(pipe_dict)
    val_metrics, thresholds, cm_dict = run_error_analysis(
        model,
        val_loader,
        pipe_dict['error_dir'],
        pipe_dict['model_dir']
    )
    generate_visual_validation_dashboard(
        lang,
        val_metrics,
        thresholds,
        history,
        cm_dict,
        REPORTS_DIR
    )
    # These imports are here because they are needed for MultimodalMemeDataset and will be used by train_language_model as well
    from transformers import AutoTokenizer, AutoImageProcessor
    from torch.utils.data import DataLoader, Dataset
    # Assuming MultimodalMemeDataset, CONFIG, HF_CACHE, device are defined in earlier cells and available globally.
    # If not, they would also need to be re-imported/re-defined here to satisfy strict isolation.
    tokenizer = AutoTokenizer.from_pretrained(
        CONFIG['indicbert_model_name'],
        cache_dir=HF_CACHE
    )
    image_processor = AutoImageProcessor.from_pretrained(
        CONFIG['siglip_model_name'],
        cache_dir=HF_CACHE
    )
    # MultimodalMemeDataset class needs to be defined for this line to work.
    # If it's not defined in an earlier cell that has been executed, this will fail.
    # Assuming it's defined and available from c6_code after execution.
    off_test_dataset = MultimodalMemeDataset(
        pipe_dict['official_test_df'],
        tokenizer,
        image_processor,
        pipe_dict['qwen_dir'],
        is_train=False,
        use_qwen=CONFIG['use_qwen_features'],
        has_labels=False
    )
    off_test_loader = DataLoader(
        off_test_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        num_workers=CONFIG['num_workers']
    )
    run_official_inference(
        model,
        off_test_loader,
        device,
        thresholds,
        pipe_dict['reverse_label_mappings'],
        lang
    )
print('\n' + '=' * 70)
print('ALL PIPELINE RUNS COMPLETED SUCCESSFULLY FOR BOTH TAMIL AND TELUGU!')
print('=' * 70)



STARTING COMPLETE HASOC PIPELINE FOR LANGUAGE: TAMIL


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

[transformers] SiglipVisionModel LOAD REPORT from: google/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.sel

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-only
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total Parameters: 375,244,304 | Initial Trainable: 375,244,304
[STAGE 1] Frozen backbones.


[TAMIL] Stage 1 Epoch 1/10:   1%|▏         | 1/70 [00:05<06:48,  5.92s/it]/tmp/ipykernel_46123/890447026.py:133: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
[TAMIL] Stage 1 Epoch 1/10: 100%|██████████| 70/70 [01:27<00:00,  1.25s/it]


Stage 1 Epoch 01 | Loss: 2.5211 | Val Mean Macro F1: 0.3361
 --> Saved Checkpoint: /content/drive/MyDrive/HASOC_Project/checkpoints/tamil/latest_checkpoint.pt
 --> Saved Checkpoint: /content/drive/MyDrive/HASOC_Project/checkpoints/tamil/best_mean_macro_f1.pt
 --> Saved Checkpoint: /content/drive/MyDrive/HASOC_Project/checkpoints/tamil/best_sentiment.pt
 --> Saved Checkpoint: /content/drive/MyDrive/HASOC_Project/checkpoints/tamil/best_sarcasm.pt
 --> Saved Checkpoint: /content/drive/MyDrive/HASOC_Project/checkpoints/tamil/best_vulgar.pt
 --> Saved Checkpoint: /content/drive/MyDrive/HASOC_Project/checkpoints/tamil/best_abuse.pt
